# Streaming Events

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) OpenAI Agents SDK roadmap.

You will learn how to use `Runner.run_streamed()` to stream agent output in real time, handle `RawResponsesStreamEvent` for tokens, and process `RunItemStreamEvent` for tool calls, messages, and handoffs.

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Basic Streaming

Use `Runner.run_streamed()` and iterate over `stream_events()` to receive tokens as they arrive.

In [ ]:
from agents import Agent, Runner

agent = Agent(
    name="Storyteller",
    instructions="You tell short, engaging stories.",
)

result = await Runner.run_streamed(agent, "Tell me a story about a robot.")

async for event in result.stream_events():
    if event.type == "raw_response_event" and hasattr(event.data, "delta"):
        print(event.data.delta, end="", flush=True)

## 4. RawResponsesStreamEvent

These events carry the raw token stream from the model. Use them to display text as it's generated.

In [ ]:
from agents.stream_events import RawResponsesStreamEvent

result = await Runner.run_streamed(agent, "Explain quantum computing in simple terms.")

async for event in result.stream_events():
    if isinstance(event, RawResponsesStreamEvent):
        if hasattr(event.data, "delta"):
            print(event.data.delta, end="", flush=True)

## 5. RunItemStreamEvent

These events represent higher-level agent actions. Each has a `name` field like `message_output_created`, `tool_called`, `tool_output`, etc.

In [ ]:
from agents import function_tool
from agents.stream_events import RunItemStreamEvent


@function_tool
def get_temperature(city: str) -> str:
    """Get the current temperature for a city."""
    temps = {"Tokyo": "18\u00b0C", "London": "12\u00b0C", "New York": "22\u00b0C"}
    return temps.get(city, "Unknown")


weather_agent = Agent(
    name="Weather Agent",
    instructions="You help users check the weather. Use the get_temperature tool.",
    tools=[get_temperature],
)

result = await Runner.run_streamed(weather_agent, "What's the temperature in Tokyo?")

async for event in result.stream_events():
    if isinstance(event, RunItemStreamEvent):
        print(f"[Event: {event.name}]")

## 6. Combining Both Event Types

Process both raw tokens (for text display) and item events (for status updates) in a single loop.

In [ ]:
result = await Runner.run_streamed(weather_agent, "What's the temperature in London?")

async for event in result.stream_events():
    if isinstance(event, RawResponsesStreamEvent):
        if hasattr(event.data, "delta"):
            print(event.data.delta, end="", flush=True)
    elif isinstance(event, RunItemStreamEvent):
        if event.name == "tool_called":
            print(f"\n[Tool called: {event.item.raw_item.name}]")
        elif event.name == "tool_output":
            print("[Tool output received]")

print(f"\n\nFinal output: {result.final_output}")

## 7. Streaming with Handoffs

When agents hand off to each other, you'll see `handoff_requested` and `handoff_occurred` events in the stream.

In [ ]:
billing_agent = Agent(name="Billing Agent", instructions="You handle billing questions.")
support_agent = Agent(
    name="Support Agent",
    instructions="You are front-line support. Hand off billing questions to the billing agent.",
    handoffs=[billing_agent],
)

result = await Runner.run_streamed(support_agent, "I have a billing question about my invoice.")

async for event in result.stream_events():
    if isinstance(event, RunItemStreamEvent):
        if event.name == "handoff_requested":
            print("[Handoff requested]")
        elif event.name == "handoff_occurred":
            print("[Handoff completed]")
    elif isinstance(event, RawResponsesStreamEvent):
        if hasattr(event.data, "delta"):
            print(event.data.delta, end="", flush=True)

print(f"\n\nFinal agent: {result.last_agent.name}")

## Key Takeaways

- Use `Runner.run_streamed()` and `stream_events()` for real-time, progressive output
- `RawResponsesStreamEvent` delivers individual tokens as the model generates them
- `RunItemStreamEvent` provides higher-level events like `tool_called`, `message_output_created`, and `handoff_occurred`
- Combine both event types for a complete streaming UI — tokens for text, items for status updates
- After streaming completes, `result.final_output` still holds the complete response